# Knowledge Graph RAG — Production-Ready Lab

This notebook builds a complete Knowledge Graph RAG pipeline step by step.
For detailed concept explanations, refer to `knowledge_graph_concepts.md`.

## Pipeline Overview

```
Phase 1: Data Ingestion          ← load, clean, chunk documents
Phase 2: KG Construction         ← extract triplets, resolve entities, build graph
Phase 3: Embedding & Indexing    ← embed nodes + chunks, store in vector DB
Phase 4: Query Engine            ← hybrid retrieval (graph + vector), LLM answer
Phase 5: Evaluation              ← relevance scoring, coverage metrics
Phase 6: Visualization           ← interactive graph, query path display
```

## Running Example
```
Domain:   A tech company's employee/project knowledge base
Entities: People, Companies, Skills, Projects, Departments
Goal:     Answer relationship questions like "Who at Acme knows Python?"
```

---
## Setup & Dependencies

In [2]:
# Install required packages (run once)
# !pip install networkx pyvis chromadb sentence-transformers nltk tiktoken

In [3]:
import os
import re
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional

print("Imports loaded.")

Imports loaded.


---
# Phase 1: Data Ingestion

**Goal:** Load raw documents → clean text → split into chunks → ready for triplet extraction.

```
Raw Documents (text, PDF, web)
        │
        ▼
   Clean & Preprocess
   (remove noise, normalize)
        │
        ▼
   Chunk with Overlap
   (split into smaller pieces so LLM can process)
        │
        ▼
   List of Chunks (ready for Phase 2)
```

## 1.1 Sample Dataset

In production, this comes from PDFs, databases, APIs, etc.
For this lab, we use a realistic company knowledge base as raw text documents.

In [4]:
# Raw documents — simulating what you'd load from files/APIs in production
# Each document is a dict with source metadata and content

RAW_DOCUMENTS = [
    {
        "source": "hr_records.txt",
        "content": """
        Alice Johnson is a Senior Software Engineer at Acme Corp. She joined in 2020 and works 
        in the Engineering department. Alice is proficient in Python, Java, and machine learning. 
        She currently leads ProjectX, which is a customer recommendation engine. Alice reports 
        to Bob Smith, who is the Engineering Manager.

        Bob Smith is the Engineering Manager at Acme Corp. He has been with the company since 2018. 
        Bob oversees the Engineering department and manages a team of 5 engineers including Alice 
        Johnson and David Lee. Bob is skilled in project management, system design, and leadership. 
        He reports to the CTO, Emily Chen.

        David Lee is a Junior Software Engineer at Acme Corp. He joined in 2023 and works in the 
        Engineering department under Bob Smith. David knows Python and JavaScript. He is currently 
        working on ProjectY, which is an internal dashboard tool.
        """
    },
    {
        "source": "project_docs.txt",
        "content": """
        ProjectX is a customer recommendation engine being developed by the Engineering department 
        at Acme Corp. The project started in Q1 2023 and is expected to be completed by Q4 2024. 
        It requires expertise in Python, machine learning, and cloud computing. Alice Johnson is 
        the project lead. The project uses AWS services including SageMaker and S3.

        ProjectY is an internal dashboard tool for monitoring company metrics. It is being developed 
        by David Lee with guidance from Alice Johnson. The project requires JavaScript, React, and 
        Python. ProjectY started in Q3 2023 and has a deadline of Q2 2024.

        ProjectZ is a data pipeline project managed by the Data Science department. Carol Martinez 
        leads this project. It requires Python, SQL, and Apache Spark. The project integrates data 
        from multiple sources into a unified data warehouse on AWS Redshift.
        """
    },
    {
        "source": "company_info.txt",
        "content": """
        Acme Corp is a technology company founded in 2015 and headquartered in San Francisco. 
        The company has three main departments: Engineering, Data Science, and Product. Emily Chen 
        is the CTO and oversees all technical departments. The company has about 50 employees.

        Carol Martinez is a Senior Data Scientist at Acme Corp. She works in the Data Science 
        department and has expertise in Python, SQL, Apache Spark, and statistical modeling. 
        Carol joined in 2019 and leads ProjectZ. She collaborates frequently with Alice Johnson 
        on cross-team data initiatives.

        Emily Chen is the Chief Technology Officer at Acme Corp. She joined the company in 2016 
        as one of the early employees. Emily has a background in distributed systems and cloud 
        architecture. She oversees both the Engineering and Data Science departments. Bob Smith 
        and Carol Martinez report directly to her.
        """
    }
]

print(f"Loaded {len(RAW_DOCUMENTS)} documents")
for doc in RAW_DOCUMENTS:
    print(f"  - {doc['source']}: {len(doc['content'])} chars")

Loaded 3 documents
  - hr_records.txt: 964 chars
  - project_docs.txt: 945 chars
  - company_info.txt: 968 chars


## 1.2 Document Data Class

A structured container for each document — holds content + metadata.
In production, this tracks where each piece of data came from (for citations and debugging).

In [5]:
@dataclass
class Document:
    """A single document with content and metadata."""
    content: str                          # the actual text
    source: str                           # where it came from (filename, URL, etc.)
    doc_id: str = ""                      # unique identifier
    metadata: Dict = field(default_factory=dict)  # any extra info

    def __post_init__(self):
        # Auto-generate a unique ID from content hash if not provided
        if not self.doc_id:
            self.doc_id = hashlib.md5(self.content.encode()).hexdigest()[:12]


@dataclass
class Chunk:
    """A piece of a document after splitting."""
    text: str                             # the chunk text
    chunk_id: str                         # unique identifier
    doc_id: str                           # which document this came from
    source: str                           # original source file
    chunk_index: int                      # position in the document (0, 1, 2, ...)
    metadata: Dict = field(default_factory=dict)


print("Data classes defined: Document, Chunk")

Data classes defined: Document, Chunk


## 1.3 Text Cleaning

Raw text is messy — extra whitespace, special characters, inconsistent formatting.
Cleaning ensures consistent input for triplet extraction later.

In [6]:
class TextCleaner:
    """Cleans and normalizes raw text for downstream processing."""

    @staticmethod
    def clean(text: str) -> str:
        text = text.strip()
        text = re.sub(r'\s+', ' ', text)           # collapse multiple whitespace into one
        text = re.sub(r'\n\s*\n', '\n', text)      # collapse multiple blank lines
        text = re.sub(r'[^\w\s.,;:!?\'\"\-()/@#&]', '', text)  # keep only useful characters
        return text


# Test it
sample = "   Alice   is   a   Senior   Engineer.  \n\n\n  She knows Python.  "
cleaned = TextCleaner.clean(sample)
print(f"Before: '{sample}'")
print(f"After:  '{cleaned}'")

Before: '   Alice   is   a   Senior   Engineer.  


  She knows Python.  '
After:  'Alice is a Senior Engineer. She knows Python.'


## 1.4 Document Loader

Takes raw document dicts → cleans them → returns structured Document objects.
In production, this would also handle PDF parsing, web scraping, API calls, etc.

In [7]:
class DocumentLoader:
    """Loads and cleans raw documents into structured Document objects."""

    def __init__(self):
        self.cleaner = TextCleaner()

    def load_from_dicts(self, raw_docs: List[Dict]) -> List[Document]:
        """Load from list of {source, content} dicts."""
        documents = []
        for raw in raw_docs:
            cleaned = self.cleaner.clean(raw["content"])
            doc = Document(
                content=cleaned,
                source=raw["source"],
                metadata={"char_count": len(cleaned), "word_count": len(cleaned.split())}
            )
            documents.append(doc)
        return documents

    def load_from_files(self, file_paths: List[str]) -> List[Document]:
        """Load from actual text files (production use)."""
        documents = []
        for path in file_paths:
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
            cleaned = self.cleaner.clean(content)
            doc = Document(
                content=cleaned,
                source=path,
                metadata={"char_count": len(cleaned), "word_count": len(cleaned.split())}
            )
            documents.append(doc)
        return documents


# Load our sample data
loader = DocumentLoader()
documents = loader.load_from_dicts(RAW_DOCUMENTS)

print(f"Loaded {len(documents)} documents:\n")
for doc in documents:
    print(f"  ID: {doc.doc_id}")
    print(f"  Source: {doc.source}")
    print(f"  Words: {doc.metadata['word_count']}")
    print(f"  Preview: {doc.content[:80]}...\n")

Loaded 3 documents:

  ID: 38faa46772e9
  Source: hr_records.txt
  Words: 141
  Preview: Alice Johnson is a Senior Software Engineer at Acme Corp. She joined in 2020 and...

  ID: 5266ae0e16b4
  Source: project_docs.txt
  Words: 135
  Preview: ProjectX is a customer recommendation engine being developed by the Engineering ...

  ID: ca0c6cc9581a
  Source: company_info.txt
  Words: 136
  Preview: Acme Corp is a technology company founded in 2015 and headquartered in San Franc...



## 1.5 Text Chunking

Documents are too long for LLM triplet extraction — we split them into smaller chunks.

**Why overlap?** A relationship like "Alice leads ProjectX" might get cut at the chunk boundary.
Overlap ensures both chunks contain the full sentence.

```
Document: [AAAA BBBB CCCC DDDD EEEE FFFF]

Without overlap (chunk_size=4):
  Chunk 1: [AAAA BBBB CCCC DDDD]
  Chunk 2: [EEEE FFFF]              ← relationship between DDDD and EEEE is LOST

With overlap (chunk_size=4, overlap=2):
  Chunk 1: [AAAA BBBB CCCC DDDD]
  Chunk 2: [CCCC DDDD EEEE FFFF]    ← CCCC DDDD appear in both → no loss
```

In [8]:
class TextChunker:
    """Splits documents into overlapping chunks by sentence boundaries."""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 100):
        self.chunk_size = chunk_size        # max characters per chunk
        self.chunk_overlap = chunk_overlap  # characters to overlap between chunks

    def _split_into_sentences(self, text: str) -> List[str]:
        """Split text into sentences using regex."""
        # Split on period, question mark, exclamation — followed by space or end
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip()]

    def chunk_document(self, document: Document) -> List[Chunk]:
        """Split a document into overlapping chunks, respecting sentence boundaries."""
        sentences = self._split_into_sentences(document.content)
        chunks = []
        current_chunk = []
        current_length = 0

        for sentence in sentences:
            # If adding this sentence exceeds chunk_size, save current chunk and start new one
            if current_length + len(sentence) > self.chunk_size and current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append(Chunk(
                    text=chunk_text,
                    chunk_id=f"{document.doc_id}_c{len(chunks)}",
                    doc_id=document.doc_id,
                    source=document.source,
                    chunk_index=len(chunks),
                    metadata={"char_count": len(chunk_text)}
                ))

                # Keep overlap — find sentences that fit within overlap size
                overlap_chunk = []
                overlap_length = 0
                for s in reversed(current_chunk):
                    if overlap_length + len(s) <= self.chunk_overlap:
                        overlap_chunk.insert(0, s)
                        overlap_length += len(s)
                    else:
                        break

                current_chunk = overlap_chunk
                current_length = overlap_length

            current_chunk.append(sentence)
            current_length += len(sentence)

        # Don't forget the last chunk
        if current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunks.append(Chunk(
                text=chunk_text,
                chunk_id=f"{document.doc_id}_c{len(chunks)}",
                doc_id=document.doc_id,
                source=document.source,
                chunk_index=len(chunks),
                metadata={"char_count": len(chunk_text)}
            ))

        return chunks

    def chunk_all(self, documents: List[Document]) -> List[Chunk]:
        """Chunk all documents."""
        all_chunks = []
        for doc in documents:
            all_chunks.extend(self.chunk_document(doc))
        return all_chunks


print("TextChunker defined.")

TextChunker defined.


In [9]:
# Chunk our documents
chunker = TextChunker(chunk_size=500, chunk_overlap=100)
chunks = chunker.chunk_all(documents)

print(f"Total chunks: {len(chunks)}\n")
print("=" * 70)
for chunk in chunks:
    print(f"\nChunk ID: {chunk.chunk_id}")
    print(f"Source:   {chunk.source}")
    print(f"Index:    {chunk.chunk_index}")
    print(f"Chars:    {chunk.metadata['char_count']}")
    print(f"Text:     {chunk.text[:120]}...")
    print("-" * 70)

Total chunks: 7


Chunk ID: 38faa46772e9_c0
Source:   hr_records.txt
Index:    0
Chars:    401
Text:     Alice Johnson is a Senior Software Engineer at Acme Corp. She joined in 2020 and works in the Engineering department. Al...
----------------------------------------------------------------------

Chunk ID: 38faa46772e9_c1
Source:   hr_records.txt
Index:    1
Chars:    472
Text:     Bob Smith is the Engineering Manager at Acme Corp. He has been with the company since 2018. Bob oversees the Engineering...
----------------------------------------------------------------------

Chunk ID: 38faa46772e9_c2
Source:   hr_records.txt
Index:    2
Chars:    108
Text:     David knows Python and JavaScript. He is currently working on ProjectY, which is an internal dashboard tool....
----------------------------------------------------------------------

Chunk ID: 5266ae0e16b4_c0
Source:   project_docs.txt
Index:    0
Chars:    484
Text:     ProjectX is a customer recommendation engine being devel

## 1.6 Phase 1 Summary — Data Ingestion Pipeline

Let's wrap everything into a single pipeline class for clean reuse.

In [10]:
class DataIngestionPipeline:
    """Complete Phase 1: Load → Clean → Chunk."""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 100):
        self.loader = DocumentLoader()
        self.chunker = TextChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    def run(self, raw_docs: List[Dict]) -> Tuple[List[Document], List[Chunk]]:
        """Run the full ingestion pipeline."""
        # Step 1: Load & clean
        documents = self.loader.load_from_dicts(raw_docs)
        print(f"[Phase 1] Loaded {len(documents)} documents")

        # Step 2: Chunk
        chunks = self.chunker.chunk_all(documents)
        print(f"[Phase 1] Created {len(chunks)} chunks (size={self.chunker.chunk_size}, overlap={self.chunker.chunk_overlap})")

        # Step 3: Summary stats
        total_chars = sum(c.metadata['char_count'] for c in chunks)
        avg_chars = total_chars // len(chunks) if chunks else 0
        print(f"[Phase 1] Total chars: {total_chars:,} | Avg per chunk: {avg_chars}")
        print(f"[Phase 1] ✅ Data ingestion complete.\n")

        return documents, chunks


# Run the full Phase 1 pipeline
pipeline = DataIngestionPipeline(chunk_size=500, chunk_overlap=100)
documents, chunks = pipeline.run(RAW_DOCUMENTS)

[Phase 1] Loaded 3 documents
[Phase 1] Created 7 chunks (size=500, overlap=100)
[Phase 1] Total chars: 2,795 | Avg per chunk: 399
[Phase 1] ✅ Data ingestion complete.



In [11]:
# Quick inspection — see what Phase 2 will receive as input
print("Chunks ready for Phase 2 (Triplet Extraction):\n")
for i, chunk in enumerate(chunks[:3]):  # show first 3
    print(f"--- Chunk {i} ({chunk.source}) ---")
    print(chunk.text)
    print()

Chunks ready for Phase 2 (Triplet Extraction):

--- Chunk 0 (hr_records.txt) ---
Alice Johnson is a Senior Software Engineer at Acme Corp. She joined in 2020 and works in the Engineering department. Alice is proficient in Python, Java, and machine learning. She currently leads ProjectX, which is a customer recommendation engine. Alice reports to Bob Smith, who is the Engineering Manager. Bob Smith is the Engineering Manager at Acme Corp. He has been with the company since 2018.

--- Chunk 1 (hr_records.txt) ---
Bob Smith is the Engineering Manager at Acme Corp. He has been with the company since 2018. Bob oversees the Engineering department and manages a team of 5 engineers including Alice Johnson and David Lee. Bob is skilled in project management, system design, and leadership. He reports to the CTO, Emily Chen. David Lee is a Junior Software Engineer at Acme Corp. He joined in 2023 and works in the Engineering department under Bob Smith. David knows Python and JavaScript.

--- Chunk

---
## Phase 1 Complete ✅

**What we built:**
- `TextCleaner` — normalizes raw text (whitespace, special chars)
- `DocumentLoader` — loads from dicts or files → structured Document objects
- `TextChunker` — splits documents into overlapping chunks at sentence boundaries
- `DataIngestionPipeline` — wraps all three into one reusable pipeline

**What we have now:**
- `documents` — list of cleaned Document objects with metadata
- `chunks` — list of Chunk objects ready for triplet extraction

**Next → Phase 2: Knowledge Graph Construction**
- Extract (Subject, Predicate, Object) triplets from each chunk
- Resolve duplicate entities ("Alice Johnson" = "Alice")
- Build the graph using NetworkX

---
# Phase 2: Knowledge Graph Construction

**Goal:** Take chunks from Phase 1 → extract triplets → resolve entities → build a NetworkX graph.

```
Chunks (from Phase 1)
        │
        ▼
   Extract Triplets (Subject, Predicate, Object)
   e.g. ("Alice Johnson", "works_at", "Acme Corp")
        │
        ▼
   Resolve Entities ("Alice" = "Alice Johnson")
        │
        ▼
   Validate & Normalize
        │
        ▼
   Build NetworkX Graph (nodes + edges)
```

## 2.1 Triplet Data Class

A triplet is the smallest unit of knowledge: **(Subject, Predicate, Object)**.

```
Sentence: "Alice Johnson is a Senior Software Engineer at Acme Corp"

Triplets extracted:
  ("Alice Johnson", "has_role",    "Senior Software Engineer")
  ("Alice Johnson", "works_at",   "Acme Corp")

Subject   = the entity doing something     (Alice Johnson)
Predicate = the relationship                (works_at)
Object    = the entity being acted upon     (Acme Corp)
```

We also track which chunk the triplet came from — needed for citations later.

In [12]:
@dataclass
class Triplet:
    """A single (Subject, Predicate, Object) knowledge triplet."""
    subject: str              # entity doing something
    predicate: str            # the relationship
    obj: str                  # entity being acted upon
    source_chunk_id: str      # which chunk this came from (for citations)
    confidence: float = 1.0   # how confident we are (1.0 = rule-matched, <1.0 = fuzzy)

    def __str__(self):
        return f"({self.subject}, {self.predicate}, {self.obj})"


# Quick test
t = Triplet("Alice Johnson", "works_at", "Acme Corp", "chunk_0")
print(f"Triplet: {t}")
print(f"Source:  {t.source_chunk_id}")
print(f"Confidence: {t.confidence}")

Triplet: (Alice Johnson, works_at, Acme Corp)
Source:  chunk_0
Confidence: 1.0


## 2.2 Rule-Based Triplet Extractor

This is the core of Phase 2 — read each chunk sentence by sentence and extract triplets using regex patterns.

**Why rule-based instead of LLM?**
- No API key needed — runs 100% locally
- Fast and deterministic — same input always gives same output
- Good enough for structured/semi-structured text (HR docs, project docs)
- In production, you'd combine this with LLM extraction for unstructured text

**How it works:**
```
Input sentence: "Alice Johnson is a Senior Software Engineer at Acme Corp"

Pattern 1: <Person> is a <Role> at <Company>
  → (Alice Johnson, has_role, Senior Software Engineer)
  → (Alice Johnson, works_at, Acme Corp)

Pattern 2: <Person> leads <Project>
  → (Alice Johnson, leads, ProjectX)

Pattern 3: <Person> reports to <Person>
  → (Alice Johnson, reports_to, Bob Smith)

Pattern 4: <Person> knows/is proficient in <Skill>
  → (Alice Johnson, has_skill, Python)
  → (Alice Johnson, has_skill, Java)
```

Each pattern is a regex that captures the subject and object from the sentence.

In [13]:
class RuleBasedTripletExtractor:
    """Extracts (Subject, Predicate, Object) triplets from text using regex patterns."""

    # Reusable regex fragments
    PERSON = r'([A-Z][a-z]+ [A-Z][a-z]+)'           # "Alice Johnson"
    ENTITY = r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)'    # "Acme Corp", "Emily Chen"
    ROLE = r'(?:a |the )?([A-Z][^.,]+?)'             # "Senior Software Engineer"
    PROJECT = r'(Project[A-Z]\w*)'                   # "ProjectX", "ProjectY"
    SKILL_LIST = r'([A-Z][\w\s,]+(?:and [\w\s]+)?)' # "Python, Java, and ML"

    def __init__(self):
        # Each pattern: (compiled_regex, handler_function)
        # Handler receives (match, chunk_id) and returns list of Triplets
        self.patterns = [
            self._pattern_role_at_company,
            self._pattern_reports_to,
            self._pattern_leads_project,
            self._pattern_works_on_project,
            self._pattern_has_skill,
            self._pattern_works_in_dept,
            self._pattern_joined_year,
            self._pattern_collaborates,
            self._pattern_project_requires,
            self._pattern_project_uses,
            self._pattern_oversees,
            self._pattern_company_headquartered,
        ]

    def _split_list(self, text: str) -> List[str]:
        """Split 'Python, Java, and machine learning' → ['Python', 'Java', 'machine learning']."""
        text = text.replace(' and ', ', ')
        return [item.strip() for item in text.split(',') if item.strip()]

    def _pattern_role_at_company(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} is {self.ROLE} at {self.ENTITY}', sent)
        if m:
            triplets.append(Triplet(m.group(1), 'has_role', m.group(2).strip(), cid))
            triplets.append(Triplet(m.group(1), 'works_at', m.group(3).strip(), cid))
        return triplets

    def _pattern_reports_to(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        for m in re.finditer(rf'{self.PERSON} reports? (?:directly )?to (?:the \w+, )?{self.PERSON}', sent):
            triplets.append(Triplet(m.group(1), 'reports_to', m.group(2), cid))
        return triplets

    def _pattern_leads_project(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        for m in re.finditer(rf'{self.PERSON} (?:leads?|is the project lead of) {self.PROJECT}', sent):
            triplets.append(Triplet(m.group(1), 'leads', m.group(2), cid))
        # Also match "<Person> leads this project" when project name is earlier in text
        for m in re.finditer(rf'{self.PERSON} (?:currently )?leads {self.PROJECT}', sent):
            triplets.append(Triplet(m.group(1), 'leads', m.group(2), cid))
        return triplets

    def _pattern_works_on_project(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        for m in re.finditer(rf'{self.PERSON} is currently working on {self.PROJECT}', sent):
            triplets.append(Triplet(m.group(1), 'works_on', m.group(2), cid))
        return triplets

    def _pattern_has_skill(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} (?:is proficient in|knows|has expertise in|is skilled in) {self.SKILL_LIST}', sent)
        if m:
            for skill in self._split_list(m.group(2)):
                triplets.append(Triplet(m.group(1), 'has_skill', skill, cid))
        return triplets

    def _pattern_works_in_dept(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} works in the (\w[\w\s]*?) department', sent)
        if m:
            triplets.append(Triplet(m.group(1), 'in_department', m.group(2).strip(), cid))
        return triplets

    def _pattern_joined_year(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} joined in (\d{{4}})', sent)
        if m:
            triplets.append(Triplet(m.group(1), 'joined_year', m.group(2), cid))
        return triplets

    def _pattern_collaborates(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} collaborates (?:frequently )?with {self.PERSON}', sent)
        if m:
            triplets.append(Triplet(m.group(1), 'collaborates_with', m.group(2), cid))
        return triplets

    def _pattern_project_requires(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PROJECT} (?:requires|requires expertise in) {self.SKILL_LIST}', sent)
        if not m:
            m = re.search(rf'(?:It|The project) requires (?:expertise in )?{self.SKILL_LIST}', sent)
            if m:
                return triplets  # can't resolve "It" without coreference
        if m and m.lastindex >= 2:
            for skill in self._split_list(m.group(2)):
                triplets.append(Triplet(m.group(1), 'requires_skill', skill, cid))
        return triplets

    def _pattern_project_uses(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PROJECT} (?:uses|is built with) {self.SKILL_LIST}', sent)
        if m:
            for tech in self._split_list(m.group(2)):
                triplets.append(Triplet(m.group(1), 'uses_tech', tech, cid))
        return triplets

    def _pattern_oversees(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.PERSON} oversees (?:the |both the )?(.+?) department', sent)
        if m:
            for dept in self._split_list(m.group(2)):
                triplets.append(Triplet(m.group(1), 'oversees', dept.strip(), cid))
        return triplets

    def _pattern_company_headquartered(self, sent: str, cid: str) -> List[Triplet]:
        triplets = []
        m = re.search(rf'{self.ENTITY} is (?:a \w+ company )?(?:founded in (\d{{4}}) and )?headquartered in {self.ENTITY}', sent)
        if m:
            triplets.append(Triplet(m.group(1), 'headquartered_in', m.group(3), cid))
            if m.group(2):
                triplets.append(Triplet(m.group(1), 'founded_year', m.group(2), cid))
        return triplets

    def extract_from_chunk(self, chunk: Chunk) -> List[Triplet]:
        """Run all patterns on every sentence in a chunk."""
        sentences = re.split(r'(?<=[.!?])\s+', chunk.text)
        triplets = []
        for sent in sentences:
            for pattern_fn in self.patterns:
                triplets.extend(pattern_fn(sent, chunk.chunk_id))
        return triplets

    def extract_all(self, chunks: List[Chunk]) -> List[Triplet]:
        """Extract triplets from all chunks."""
        all_triplets = []
        for chunk in chunks:
            all_triplets.extend(self.extract_from_chunk(chunk))
        return all_triplets


print("RuleBasedTripletExtractor defined.")

RuleBasedTripletExtractor defined.


In [14]:
# Extract triplets from all chunks
extractor = RuleBasedTripletExtractor()
raw_triplets = extractor.extract_all(chunks)

print(f"Extracted {len(raw_triplets)} raw triplets:\n")
for t in raw_triplets:
    print(f"  {t}  ← from {t.source_chunk_id}")

Extracted 14 raw triplets:

  (Alice Johnson, has_role, Senior Software Engineer)  ← from 38faa46772e9_c0
  (Alice Johnson, works_at, Acme Corp)  ← from 38faa46772e9_c0
  (Bob Smith, has_role, Engineering Manager)  ← from 38faa46772e9_c0
  (Bob Smith, works_at, Acme Corp)  ← from 38faa46772e9_c0
  (Bob Smith, has_role, Engineering Manager)  ← from 38faa46772e9_c1
  (Bob Smith, works_at, Acme Corp)  ← from 38faa46772e9_c1
  (David Lee, has_role, Junior Software Engineer)  ← from 38faa46772e9_c1
  (David Lee, works_at, Acme Corp)  ← from 38faa46772e9_c1
  (Acme Corp, headquartered_in, San Francisco)  ← from ca0c6cc9581a_c0
  (Acme Corp, founded_year, 2015)  ← from ca0c6cc9581a_c0
  (Carol Martinez, has_role, Senior Data Scientist)  ← from ca0c6cc9581a_c0
  (Carol Martinez, works_at, Acme Corp)  ← from ca0c6cc9581a_c0
  (Emily Chen, has_role, Chief Technology Officer)  ← from ca0c6cc9581a_c1
  (Emily Chen, works_at, Acme Corp)  ← from ca0c6cc9581a_c1


## 2.3 Entity Resolver

The same entity can appear with different names across chunks:

```
Chunk 1: "Alice Johnson is a Senior Software Engineer"  → subject = "Alice Johnson"
Chunk 2: "Alice is proficient in Python"                → subject = "Alice"
Chunk 3: "alice johnson leads ProjectX"                 → subject = "alice johnson"

Without resolution: 3 separate nodes in the graph (bad!)
With resolution:    1 node "Alice Johnson" with all edges (good!)
```

**Strategy:**
1. Build an alias map from all entity names seen in triplets
2. Group names that share a word ("Alice Johnson" and "Alice" share "Alice")
3. Pick the longest name as the canonical form
4. Replace all aliases in triplets with the canonical name

In [15]:
class EntityResolver:
    """Merges duplicate entities like 'Alice Johnson' / 'Alice' / 'alice johnson'."""

    def __init__(self):
        self.alias_map: Dict[str, str] = {}  # alias → canonical name

    def _normalize(self, name: str) -> str:
        """Lowercase + strip for comparison."""
        return name.strip().lower()

    def _title_case(self, name: str) -> str:
        """Convert to title case, but keep known acronyms uppercase."""
        acronyms = {'cto', 'ceo', 'aws', 'sql', 'api'}
        words = name.split()
        return ' '.join(w.upper() if w.lower() in acronyms else w.title() for w in words)

    def build_alias_map(self, triplets: List[Triplet]) -> Dict[str, str]:
        """Scan all entities in triplets, group by shared words, pick longest as canonical."""
        # Collect all unique entity names
        all_names = set()
        for t in triplets:
            all_names.add(t.subject)
            all_names.add(t.obj)

        # Group: for each name, find if it's a substring of a longer name
        sorted_names = sorted(all_names, key=len, reverse=True)  # longest first
        alias_map = {}

        for name in sorted_names:
            norm = self._normalize(name)
            if norm in alias_map:
                continue

            # This is a potential canonical name — find shorter aliases
            canonical = self._title_case(name)
            alias_map[norm] = canonical

            # Check if any other name is a substring (first or last name match)
            canonical_words = set(norm.split())
            for other in sorted_names:
                other_norm = self._normalize(other)
                if other_norm == norm or other_norm in alias_map:
                    continue
                other_words = set(other_norm.split())
                # Single-word name that matches one word of a multi-word name
                if len(other_words) == 1 and other_words.issubset(canonical_words) and len(canonical_words) > 1:
                    alias_map[other_norm] = canonical

        self.alias_map = alias_map
        return alias_map

    def resolve(self, name: str) -> str:
        """Look up canonical name for a given entity."""
        norm = self._normalize(name)
        return self.alias_map.get(norm, self._title_case(name))

    def resolve_triplets(self, triplets: List[Triplet]) -> List[Triplet]:
        """Replace subject and object in all triplets with canonical names."""
        self.build_alias_map(triplets)
        resolved = []
        for t in triplets:
            resolved.append(Triplet(
                subject=self.resolve(t.subject),
                predicate=t.predicate,
                obj=self.resolve(t.obj),
                source_chunk_id=t.source_chunk_id,
                confidence=t.confidence
            ))
        return resolved


print("EntityResolver defined.")

EntityResolver defined.


In [16]:
# Resolve entities
resolver = EntityResolver()
resolved_triplets = resolver.resolve_triplets(raw_triplets)

# Show alias map
print("Alias Map (what got merged):\n")
for alias, canonical in sorted(resolver.alias_map.items()):
    if alias != canonical.lower():
        print(f"  '{alias}' → '{canonical}'")

print(f"\nResolved {len(resolved_triplets)} triplets:\n")
for t in resolved_triplets:
    print(f"  {t}")

Alias Map (what got merged):


Resolved 14 triplets:

  (Alice Johnson, has_role, Senior Software Engineer)
  (Alice Johnson, works_at, Acme Corp)
  (Bob Smith, has_role, Engineering Manager)
  (Bob Smith, works_at, Acme Corp)
  (Bob Smith, has_role, Engineering Manager)
  (Bob Smith, works_at, Acme Corp)
  (David Lee, has_role, Junior Software Engineer)
  (David Lee, works_at, Acme Corp)
  (Acme Corp, headquartered_in, San Francisco)
  (Acme Corp, founded_year, 2015)
  (Carol Martinez, has_role, Senior Data Scientist)
  (Carol Martinez, works_at, Acme Corp)
  (Emily Chen, has_role, Chief Technology Officer)
  (Emily Chen, works_at, Acme Corp)


## 2.4 Triplet Validator & Deduplicator

After extraction + entity resolution, we often get:
- **Duplicates** — same triplet extracted from overlapping chunks
- **Invalid triplets** — empty subject/object, subject == object, too-short names

```
Before validation:                          After validation:
  (Alice Johnson, works_at, Acme Corp)        (Alice Johnson, works_at, Acme Corp)  ← kept
  (Alice Johnson, works_at, Acme Corp)        (removed — duplicate)
  (, has_skill, Python)                       (removed — empty subject)
  (Alice Johnson, has_skill, Alice Johnson)   (removed — subject == object)
```

In [17]:
class TripletValidator:
    """Removes invalid and duplicate triplets."""

    @staticmethod
    def is_valid(t: Triplet) -> bool:
        if not t.subject.strip() or not t.obj.strip():
            return False
        if t.subject.strip().lower() == t.obj.strip().lower():
            return False
        if len(t.subject.strip()) < 2 or len(t.obj.strip()) < 2:
            return False
        return True

    @staticmethod
    def deduplicate(triplets: List[Triplet]) -> List[Triplet]:
        seen = set()
        unique = []
        for t in triplets:
            key = (t.subject.lower(), t.predicate, t.obj.lower())
            if key not in seen:
                seen.add(key)
                unique.append(t)
        return unique

    def validate(self, triplets: List[Triplet]) -> List[Triplet]:
        valid = [t for t in triplets if self.is_valid(t)]
        deduped = self.deduplicate(valid)
        return deduped


print("TripletValidator defined.")

TripletValidator defined.


In [18]:
# Validate and deduplicate
validator = TripletValidator()
clean_triplets = validator.validate(resolved_triplets)

removed = len(resolved_triplets) - len(clean_triplets)
print(f"Before: {len(resolved_triplets)} triplets")
print(f"After:  {len(clean_triplets)} triplets ({removed} removed)\n")

for t in clean_triplets:
    print(f"  {t}")

Before: 14 triplets
After:  12 triplets (2 removed)

  (Alice Johnson, has_role, Senior Software Engineer)
  (Alice Johnson, works_at, Acme Corp)
  (Bob Smith, has_role, Engineering Manager)
  (Bob Smith, works_at, Acme Corp)
  (David Lee, has_role, Junior Software Engineer)
  (David Lee, works_at, Acme Corp)
  (Acme Corp, headquartered_in, San Francisco)
  (Acme Corp, founded_year, 2015)
  (Carol Martinez, has_role, Senior Data Scientist)
  (Carol Martinez, works_at, Acme Corp)
  (Emily Chen, has_role, Chief Technology Officer)
  (Emily Chen, works_at, Acme Corp)


## 2.5 Build NetworkX Graph

Now we convert our clean triplets into an actual graph data structure using NetworkX.

```
Triplet: (Alice Johnson, works_at, Acme Corp)

Becomes:
  Node: "Alice Johnson"  ─── works_at ───▶  Node: "Acme Corp"

Each node stores:
  - label (the entity name)
  - type (auto-detected: Person, Company, Project, Skill, Department, Other)

Each edge stores:
  - predicate (the relationship type)
  - source_chunk_id (for citations)
  - confidence score
```

In [19]:
import networkx as nx

class KnowledgeGraphBuilder:
    """Builds a NetworkX directed graph from validated triplets."""

    # Simple heuristic to detect entity type from name or predicate context
    PERSON_PREDICATES = {'has_role', 'reports_to', 'has_skill', 'joined_year', 'in_department', 'collaborates_with'}
    PROJECT_PATTERN = re.compile(r'^Project[A-Z]')
    SKILL_KEYWORDS = {'python', 'java', 'javascript', 'react', 'sql', 'machine learning',
                      'cloud computing', 'apache spark', 'statistical modeling',
                      'project management', 'system design', 'leadership',
                      'distributed systems', 'cloud architecture'}
    DEPT_KEYWORDS = {'engineering', 'data science', 'product'}

    def __init__(self):
        self.graph = nx.DiGraph()

    def _detect_type(self, name: str, predicate: str = '', is_subject: bool = True) -> str:
        """Guess entity type from name and context."""
        name_lower = name.lower()
        if self.PROJECT_PATTERN.match(name):
            return 'Project'
        if name_lower in self.SKILL_KEYWORDS:
            return 'Skill'
        if name_lower in self.DEPT_KEYWORDS:
            return 'Department'
        if predicate in ('works_at', 'headquartered_in', 'founded_year'):
            return 'Person' if is_subject else 'Company'
        if predicate in self.PERSON_PREDICATES:
            return 'Person'
        if predicate == 'has_skill':
            return 'Skill' if not is_subject else 'Person'
        if 'corp' in name_lower or 'inc' in name_lower or 'company' in name_lower:
            return 'Company'
        return 'Other'

    def build(self, triplets: List[Triplet]) -> nx.DiGraph:
        """Convert triplets to a directed graph."""
        for t in triplets:
            # Add/update subject node
            sub_type = self._detect_type(t.subject, t.predicate, is_subject=True)
            if t.subject not in self.graph:
                self.graph.add_node(t.subject, type=sub_type)

            # Add/update object node
            obj_type = self._detect_type(t.obj, t.predicate, is_subject=False)
            if t.obj not in self.graph:
                self.graph.add_node(t.obj, type=obj_type)

            # Add edge
            self.graph.add_edge(
                t.subject, t.obj,
                predicate=t.predicate,
                source_chunk_id=t.source_chunk_id,
                confidence=t.confidence
            )

        return self.graph


print("KnowledgeGraphBuilder defined.")

KnowledgeGraphBuilder defined.


In [20]:
# Build the graph
builder = KnowledgeGraphBuilder()
kg = builder.build(clean_triplets)

print(f"Knowledge Graph built!")
print(f"  Nodes: {kg.number_of_nodes()}")
print(f"  Edges: {kg.number_of_edges()}")

print(f"\nNodes by type:")
from collections import Counter
type_counts = Counter(data.get('type', 'Other') for _, data in kg.nodes(data=True))
for t, count in type_counts.most_common():
    print(f"  {t}: {count}")

print(f"\nAll nodes:")
for node, data in kg.nodes(data=True):
    print(f"  [{data.get('type', '?')}] {node}")

print(f"\nAll edges:")
for u, v, data in kg.edges(data=True):
    print(f"  {u} ── {data['predicate']} ──▶ {v}")

Knowledge Graph built!
  Nodes: 13
  Edges: 12

Nodes by type:
  Person: 10
  Company: 3

All nodes:
  [Person] Alice Johnson
  [Person] Senior Software Engineer
  [Company] Acme Corp
  [Person] Bob Smith
  [Person] Engineering Manager
  [Person] David Lee
  [Person] Junior Software Engineer
  [Company] San Francisco
  [Company] 2015
  [Person] Carol Martinez
  [Person] Senior Data Scientist
  [Person] Emily Chen
  [Person] Chief Technology Officer

All edges:
  Alice Johnson ── has_role ──▶ Senior Software Engineer
  Alice Johnson ── works_at ──▶ Acme Corp
  Acme Corp ── headquartered_in ──▶ San Francisco
  Acme Corp ── founded_year ──▶ 2015
  Bob Smith ── has_role ──▶ Engineering Manager
  Bob Smith ── works_at ──▶ Acme Corp
  David Lee ── has_role ──▶ Junior Software Engineer
  David Lee ── works_at ──▶ Acme Corp
  Carol Martinez ── has_role ──▶ Senior Data Scientist
  Carol Martinez ── works_at ──▶ Acme Corp
  Emily Chen ── has_role ──▶ Chief Technology Officer
  Emily Chen ── work

## 2.6 Phase 2 Summary — KG Construction Pipeline

Wrap all Phase 2 steps into a single reusable pipeline class.

In [21]:
class KGConstructionPipeline:
    """Complete Phase 2: Extract triplets → Resolve entities → Validate → Build graph."""

    def __init__(self):
        self.extractor = RuleBasedTripletExtractor()
        self.resolver = EntityResolver()
        self.validator = TripletValidator()
        self.builder = KnowledgeGraphBuilder()

    def run(self, chunks: List[Chunk]) -> Tuple[nx.DiGraph, List[Triplet]]:
        """Run the full KG construction pipeline."""
        # Step 1: Extract
        raw = self.extractor.extract_all(chunks)
        print(f"[Phase 2] Extracted {len(raw)} raw triplets")

        # Step 2: Resolve entities
        resolved = self.resolver.resolve_triplets(raw)
        merged = sum(1 for a, c in self.resolver.alias_map.items() if a != c.lower())
        print(f"[Phase 2] Resolved entities ({merged} aliases merged)")

        # Step 3: Validate & deduplicate
        clean = self.validator.validate(resolved)
        print(f"[Phase 2] Validated: {len(clean)} triplets ({len(resolved) - len(clean)} removed)")

        # Step 4: Build graph
        graph = self.builder.build(clean)
        print(f"[Phase 2] Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
        print(f"[Phase 2] ✅ KG construction complete.\n")

        return graph, clean


# Run the full Phase 2 pipeline
kg_pipeline = KGConstructionPipeline()
kg, clean_triplets = kg_pipeline.run(chunks)

[Phase 2] Extracted 14 raw triplets
[Phase 2] Resolved entities (0 aliases merged)
[Phase 2] Validated: 12 triplets (2 removed)
[Phase 2] Graph: 13 nodes, 12 edges
[Phase 2] ✅ KG construction complete.



In [22]:
# Quick graph queries — preview what Phase 4 (Query Engine) will use

print("=== Sample Graph Queries ===\n")

# Who does Alice Johnson work with?
alice_edges = list(kg.edges('Alice Johnson', data=True))
print("Alice Johnson's relationships:")
for _, target, data in alice_edges:
    print(f"  ── {data['predicate']} ──▶ {target}")

# Who has Python skill?
print("\nWho has Python skill?")
for u, v, data in kg.edges(data=True):
    if data['predicate'] == 'has_skill' and v == 'Python':
        print(f"  {u}")

# What are the neighbors of Acme Corp?
print("\nEntities connected to Acme Corp:")
for neighbor in kg.predecessors('Acme Corp'):
    edge_data = kg.get_edge_data(neighbor, 'Acme Corp')
    print(f"  {neighbor} ── {edge_data['predicate']} ──▶ Acme Corp")

=== Sample Graph Queries ===

Alice Johnson's relationships:
  ── has_role ──▶ Senior Software Engineer
  ── works_at ──▶ Acme Corp

Who has Python skill?

Entities connected to Acme Corp:
  Alice Johnson ── works_at ──▶ Acme Corp
  Bob Smith ── works_at ──▶ Acme Corp
  David Lee ── works_at ──▶ Acme Corp
  Carol Martinez ── works_at ──▶ Acme Corp
  Emily Chen ── works_at ──▶ Acme Corp


---
## Phase 2 Complete ✅

**What we built:**
- `Triplet` — dataclass for (Subject, Predicate, Object) with source tracking
- `RuleBasedTripletExtractor` — 12 regex patterns to extract relationships from text
- `EntityResolver` — merges duplicate entity names ("Alice" → "Alice Johnson")
- `TripletValidator` — removes invalid/duplicate triplets
- `KnowledgeGraphBuilder` — converts triplets to a NetworkX directed graph with typed nodes
- `KGConstructionPipeline` — wraps all five into one reusable pipeline

**What we have now:**
- `kg` — a NetworkX DiGraph with entities as nodes and relationships as edges
- `clean_triplets` — validated triplet list (used for embedding in Phase 3)

**Next → Phase 3: Embedding & Indexing**
- Generate embeddings for graph nodes + text chunks
- Store in ChromaDB for vector similarity search

---
# Phase 3: Embedding & Indexing

**Goal:** Convert graph nodes and text chunks into vectors → store in ChromaDB for fast similarity search.

```
KG nodes + Chunks (from Phase 1 & 2)
        │
        ▼
   Generate Embeddings (sentence-transformers, runs locally)
   "Alice Johnson" → [0.12, -0.34, 0.56, ...]  (384 dimensions)
        │
        ▼
   Store in ChromaDB
   Collection 1: chunk_embeddings   (for text similarity)
   Collection 2: node_embeddings    (for entity similarity)
        │
        ▼
   Ready for Phase 4 (hybrid retrieval)
```

**Why two collections?**
- chunk_embeddings: find relevant text passages (like normal RAG)
- node_embeddings: find related entities in the graph (KG-specific)

## 3.1 Embedding Model

We use `sentence-transformers/all-MiniLM-L6-v2` — a small, fast model that runs locally.

```
Input:  "Alice Johnson is a Senior Software Engineer"
Output: [0.12, -0.34, 0.56, ...]  → 384-dimensional vector

Similar texts → vectors close together in 384D space
  "Alice is an engineer"     ≈  "Alice Johnson is a Senior Software Engineer"
  "The weather is nice today" ≠  "Alice Johnson is a Senior Software Engineer"
```

In production, you'd use a larger model (e.g. `all-mpnet-base-v2` with 768 dims) for better accuracy.

In [23]:
from sentence_transformers import SentenceTransformer

class EmbeddingModel:
    """Wrapper around sentence-transformers for generating embeddings."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
        print(f"Loaded '{model_name}' — {self.dim} dimensions")

    def embed(self, texts: List[str]) -> List[List[float]]:
        """Convert list of strings to list of embedding vectors."""
        embeddings = self.model.encode(texts, show_progress_bar=False)
        return embeddings.tolist()


# Initialize
embed_model = EmbeddingModel()

# Quick test
test_emb = embed_model.embed(['Alice Johnson is a Senior Software Engineer'])
print(f"Test embedding: {len(test_emb[0])} dims, first 5 values: {test_emb[0][:5]}")

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 'all-MiniLM-L6-v2' — 384 dimensions
Test embedding: 384 dims, first 5 values: [-0.060140401124954224, -0.03693591430783272, -0.025882143527269363, 0.035577286034822464, -0.00518836872652173]


## 3.2 Prepare Texts for Embedding

We need to embed two things:

1. **Chunks** — the raw text passages (same as normal RAG)
2. **Graph nodes** — each entity gets a text description built from its edges

```
Node: "Alice Johnson"
Edges: works_at→Acme Corp, has_role→Senior Software Engineer, has_skill→Python, leads→ProjectX

Node description (what we embed):
  "Alice Johnson: works_at Acme Corp. has_role Senior Software Engineer. has_skill Python. leads ProjectX."
```

This way, searching for "Python engineer" finds the Alice Johnson node.

In [24]:
def build_node_descriptions(graph: nx.DiGraph) -> Dict[str, str]:
    """Build a text description for each node from its edges."""
    descriptions = {}
    for node in graph.nodes():
        parts = [f"{node}:"]
        # Outgoing edges
        for _, target, data in graph.out_edges(node, data=True):
            parts.append(f"{data['predicate']} {target}.")
        # Incoming edges
        for source, _, data in graph.in_edges(node, data=True):
            parts.append(f"{source} {data['predicate']} this.")
        descriptions[node] = ' '.join(parts)
    return descriptions


# Build descriptions
node_descriptions = build_node_descriptions(kg)

print("Node descriptions (what gets embedded):\n")
for node, desc in list(node_descriptions.items())[:5]:
    print(f"  {node}:")
    print(f"    {desc[:120]}...")
    print()

Node descriptions (what gets embedded):

  Alice Johnson:
    Alice Johnson: has_role Senior Software Engineer. works_at Acme Corp....

  Senior Software Engineer:
    Senior Software Engineer: Alice Johnson has_role this....

  Acme Corp:
    Acme Corp: headquartered_in San Francisco. founded_year 2015. Alice Johnson works_at this. Bob Smith works_at this. Davi...

  Bob Smith:
    Bob Smith: has_role Engineering Manager. works_at Acme Corp....

  Engineering Manager:
    Engineering Manager: Bob Smith has_role this....



## 3.3 ChromaDB Vector Store

ChromaDB stores embeddings + metadata and lets us search by similarity.

```
Store:                              Query:
  ID: chunk_0                         "Who knows Python?"
  Text: "Alice is proficient..."           │
  Vector: [0.12, -0.34, ...]            ▼
                                    embed("Who knows Python?")
                                    = [0.11, -0.30, ...]
                                           │
                                           ▼
                                    cosine similarity search
                                           │
                                           ▼
                                    Top-K most similar chunks
```

In [25]:
import chromadb

class VectorStore:
    """ChromaDB wrapper for storing and querying embeddings."""

    def __init__(self, persist_dir: str = './chroma_db'):
        self.client = chromadb.Client()  # in-memory for lab; use PersistentClient in production

    def create_collection(self, name: str) -> chromadb.Collection:
        """Create or get a collection."""
        return self.client.get_or_create_collection(
            name=name,
            metadata={'hnsw:space': 'cosine'}
        )

    def add_documents(self, collection: chromadb.Collection,
                      ids: List[str], texts: List[str],
                      embeddings: List[List[float]],
                      metadatas: List[Dict] = None):
        """Add documents with embeddings to a collection."""
        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas or [{} for _ in ids]
        )

    def query(self, collection: chromadb.Collection,
             query_embedding: List[float],
             top_k: int = 5) -> Dict:
        """Find top-K most similar documents."""
        return collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=['documents', 'metadatas', 'distances']
        )


print("VectorStore defined.")

VectorStore defined.


## 3.4 Index Chunks & Nodes

Now we embed everything and store it in ChromaDB — two separate collections.

In [26]:
# Initialize vector store
vector_store = VectorStore()

# --- Collection 1: Chunk embeddings ---
chunk_collection = vector_store.create_collection('chunks')

chunk_texts = [c.text for c in chunks]
chunk_ids = [c.chunk_id for c in chunks]
chunk_metas = [{'source': c.source, 'doc_id': c.doc_id, 'chunk_index': c.chunk_index} for c in chunks]
chunk_embeddings = embed_model.embed(chunk_texts)

vector_store.add_documents(chunk_collection, chunk_ids, chunk_texts, chunk_embeddings, chunk_metas)
print(f"Indexed {len(chunk_ids)} chunks into 'chunks' collection")

# --- Collection 2: Node embeddings ---
node_collection = vector_store.create_collection('nodes')

node_names = list(node_descriptions.keys())
node_texts = list(node_descriptions.values())
node_ids = [f"node_{i}" for i in range(len(node_names))]
node_metas = [{'name': name, 'type': kg.nodes[name].get('type', 'Other')} for name in node_names]
node_embeddings = embed_model.embed(node_texts)

vector_store.add_documents(node_collection, node_ids, node_texts, node_embeddings, node_metas)
print(f"Indexed {len(node_ids)} nodes into 'nodes' collection")

Indexed 7 chunks into 'chunks' collection
Indexed 13 nodes into 'nodes' collection


## 3.5 Test Retrieval

Quick sanity check — search both collections to make sure embeddings work.

In [27]:
# Test query
test_query = "Who knows Python at Acme?"
query_emb = embed_model.embed([test_query])[0]

print(f"Query: '{test_query}'\n")

# Search chunks
chunk_results = vector_store.query(chunk_collection, query_emb, top_k=3)
print("Top 3 chunks:")
for i, (doc, dist) in enumerate(zip(chunk_results['documents'][0], chunk_results['distances'][0])):
    print(f"  {i+1}. (dist={dist:.3f}) {doc[:100]}...")

# Search nodes
node_results = vector_store.query(node_collection, query_emb, top_k=3)
print(f"\nTop 3 nodes:")
for i, (meta, dist) in enumerate(zip(node_results['metadatas'][0], node_results['distances'][0])):
    print(f"  {i+1}. (dist={dist:.3f}) [{meta['type']}] {meta['name']}")

Query: 'Who knows Python at Acme?'

Top 3 chunks:
  1. (dist=0.457) Acme Corp is a technology company founded in 2015 and headquartered in San Francisco. The company ha...
  2. (dist=0.505) David knows Python and JavaScript. He is currently working on ProjectY, which is an internal dashboa...
  3. (dist=0.556) Alice Johnson is a Senior Software Engineer at Acme Corp. She joined in 2020 and works in the Engine...

Top 3 nodes:
  1. (dist=0.547) [Person] David Lee
  2. (dist=0.553) [Person] Alice Johnson
  3. (dist=0.574) [Person] Emily Chen


## 3.6 Phase 3 Summary — Embedding & Indexing Pipeline

Wrap everything into a reusable pipeline.

In [28]:
class EmbeddingIndexingPipeline:
    """Complete Phase 3: Embed chunks + nodes → Store in ChromaDB."""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.embed_model = EmbeddingModel(model_name)
        self.vector_store = VectorStore()

    def run(self, chunks: List[Chunk], graph: nx.DiGraph) -> Tuple[chromadb.Collection, chromadb.Collection]:
        """Run the full embedding + indexing pipeline."""
        # Step 1: Embed & index chunks
        chunk_col = self.vector_store.create_collection('chunks')
        chunk_texts = [c.text for c in chunks]
        chunk_embs = self.embed_model.embed(chunk_texts)
        chunk_metas = [{'source': c.source, 'doc_id': c.doc_id, 'chunk_index': c.chunk_index} for c in chunks]
        self.vector_store.add_documents(chunk_col, [c.chunk_id for c in chunks], chunk_texts, chunk_embs, chunk_metas)
        print(f"[Phase 3] Indexed {len(chunks)} chunks")

        # Step 2: Embed & index nodes
        node_col = self.vector_store.create_collection('nodes')
        node_descs = build_node_descriptions(graph)
        names = list(node_descs.keys())
        texts = list(node_descs.values())
        embs = self.embed_model.embed(texts)
        metas = [{'name': n, 'type': graph.nodes[n].get('type', 'Other')} for n in names]
        self.vector_store.add_documents(node_col, [f'node_{i}' for i in range(len(names))], texts, embs, metas)
        print(f"[Phase 3] Indexed {len(names)} nodes")
        print(f"[Phase 3] ✅ Embedding & indexing complete.\n")

        return chunk_col, node_col


print("EmbeddingIndexingPipeline defined.")

EmbeddingIndexingPipeline defined.


---
## Phase 3 Complete ✅

**What we built:**
- `EmbeddingModel` — wrapper around sentence-transformers (384-dim vectors, runs locally)
- `build_node_descriptions` — converts graph nodes into embeddable text descriptions
- `VectorStore` — ChromaDB wrapper for storing/querying embeddings
- `EmbeddingIndexingPipeline` — wraps all three into one reusable pipeline

**What we have now:**
- `chunk_collection` — ChromaDB collection with chunk embeddings
- `node_collection` — ChromaDB collection with node embeddings
- Both searchable by cosine similarity

**Next → Phase 4: Query Engine**
- Hybrid retrieval: graph traversal + vector similarity
- Combine results and generate final answer

---
# Phase 4: Query Engine

**Goal:** Take a user question → retrieve from both graph + vector store → combine → generate answer.

```
User Query: "Who at Acme knows Python?"
        │
        ├─── Graph Retrieval (traverse KG)
        │     Find nodes connected to "Python" via "has_skill"
        │     → Alice Johnson, David Lee, Carol Martinez
        │
        ├─── Vector Retrieval (search ChromaDB)
        │     Find chunks similar to "Who at Acme knows Python?"
        │     → top-3 relevant text passages
        │
        ▼
   Combine both results → build context → generate answer
```

**Why hybrid?** Graph gives precise structured facts. Vector gives fuzzy text matches.
Together they cover both exact relationships and contextual information.

## 4.1 Graph Retriever

Searches the KG by finding entities mentioned in the query, then traversing their edges.

```
Query: "Who knows Python?"

Step 1: Find "Python" in graph nodes → found!
Step 2: Get all edges pointing TO "Python"
        Alice Johnson ── has_skill ──▶ Python
        David Lee     ── has_skill ──▶ Python
        Carol Martinez── has_skill ──▶ Python
Step 3: Return these as structured facts
```

In [29]:
class GraphRetriever:
    """Retrieves relevant facts from the knowledge graph."""

    def __init__(self, graph: nx.DiGraph):
        self.graph = graph
        # Build a lowercase lookup for node matching
        self.node_lookup = {n.lower(): n for n in graph.nodes()}

    def _find_entities_in_query(self, query: str) -> List[str]:
        """Find graph nodes mentioned in the query."""
        query_lower = query.lower()
        found = []
        # Check each node name against the query (longest first to avoid partial matches)
        for name_lower, name in sorted(self.node_lookup.items(), key=lambda x: len(x[0]), reverse=True):
            if name_lower in query_lower:
                found.append(name)
        return found

    def _get_node_facts(self, node: str, max_hops: int = 2) -> List[str]:
        """Get all facts within max_hops of a node."""
        facts = []
        visited = set()
        queue = [(node, 0)]

        while queue:
            current, depth = queue.pop(0)
            if current in visited or depth > max_hops:
                continue
            visited.add(current)

            # Outgoing edges
            for _, target, data in self.graph.out_edges(current, data=True):
                facts.append(f"{current} {data['predicate']} {target}")
                if depth + 1 <= max_hops:
                    queue.append((target, depth + 1))

            # Incoming edges
            for source, _, data in self.graph.in_edges(current, data=True):
                facts.append(f"{source} {data['predicate']} {current}")
                if depth + 1 <= max_hops:
                    queue.append((source, depth + 1))

        return list(set(facts))  # deduplicate

    def retrieve(self, query: str, max_hops: int = 2) -> List[str]:
        """Find entities in query, then get their neighborhood facts."""
        entities = self._find_entities_in_query(query)
        all_facts = []
        for entity in entities:
            all_facts.extend(self._get_node_facts(entity, max_hops))
        return list(set(all_facts))


print("GraphRetriever defined.")

GraphRetriever defined.


In [30]:
# Test graph retrieval
graph_retriever = GraphRetriever(kg)

test_q = "Who knows Python?"
facts = graph_retriever.retrieve(test_q, max_hops=1)

print(f"Query: '{test_q}'\n")
print(f"Entities found: {graph_retriever._find_entities_in_query(test_q)}")
print(f"\nGraph facts ({len(facts)}):")
for f in facts:
    print(f"  • {f}")

Query: 'Who knows Python?'

Entities found: []

Graph facts (0):


## 4.2 Vector Retriever

Searches ChromaDB for chunks and nodes similar to the query embedding.

In [31]:
class VectorRetriever:
    """Retrieves relevant chunks and nodes from ChromaDB."""

    def __init__(self, embed_model: EmbeddingModel,
                 chunk_collection: chromadb.Collection,
                 node_collection: chromadb.Collection,
                 vector_store: VectorStore):
        self.embed_model = embed_model
        self.chunk_col = chunk_collection
        self.node_col = node_collection
        self.vector_store = vector_store

    def retrieve(self, query: str, top_k_chunks: int = 3, top_k_nodes: int = 3) -> Dict:
        """Search both collections and return combined results."""
        query_emb = self.embed_model.embed([query])[0]

        chunk_results = self.vector_store.query(self.chunk_col, query_emb, top_k_chunks)
        node_results = self.vector_store.query(self.node_col, query_emb, top_k_nodes)

        return {
            'chunks': chunk_results['documents'][0] if chunk_results['documents'] else [],
            'chunk_distances': chunk_results['distances'][0] if chunk_results['distances'] else [],
            'nodes': node_results['metadatas'][0] if node_results['metadatas'] else [],
            'node_distances': node_results['distances'][0] if node_results['distances'] else [],
        }


print("VectorRetriever defined.")

VectorRetriever defined.


In [32]:
# Test vector retrieval
vector_retriever = VectorRetriever(embed_model, chunk_collection, node_collection, vector_store)

vr = vector_retriever.retrieve("Who knows Python?")

print("Vector retrieval results:\n")
print("Top chunks:")
for i, (chunk, dist) in enumerate(zip(vr['chunks'], vr['chunk_distances'])):
    print(f"  {i+1}. (dist={dist:.3f}) {chunk[:80]}...")

print("\nTop nodes:")
for i, (meta, dist) in enumerate(zip(vr['nodes'], vr['node_distances'])):
    print(f"  {i+1}. (dist={dist:.3f}) [{meta['type']}] {meta['name']}")

Vector retrieval results:

Top chunks:
  1. (dist=0.443) David knows Python and JavaScript. He is currently working on ProjectY, which is...
  2. (dist=0.623) It is being developed by David Lee with guidance from Alice Johnson. The project...
  3. (dist=0.661) ProjectX is a customer recommendation engine being developed by the Engineering ...

Top nodes:
  1. (dist=0.725) [Person] David Lee
  2. (dist=0.744) [Person] Junior Software Engineer
  3. (dist=0.745) [Person] Senior Software Engineer


## 4.3 Hybrid Query Engine

Combines graph facts + vector results into a single context, then generates an answer.

```
Query: "Who at Acme knows Python?"

Graph facts:                          Vector chunks:
  Alice Johnson has_skill Python        "Alice is proficient in Python..."
  David Lee has_skill Python            "David knows Python and JavaScript..."
  Carol Martinez has_skill Python       "Carol has expertise in Python..."
                    │                              │
                    └──────────────┬───────────────┘
                               │
                        Combined Context
                               │
                        Generate Answer
```

For this lab, we generate the answer using template-based logic (no LLM API needed).
In production, you'd pass the context to an LLM like Claude or GPT.

In [33]:
class HybridQueryEngine:
    """Combines graph + vector retrieval to answer questions."""

    def __init__(self, graph_retriever: GraphRetriever, vector_retriever: VectorRetriever):
        self.graph_retriever = graph_retriever
        self.vector_retriever = vector_retriever

    def _build_context(self, graph_facts: List[str], vector_results: Dict) -> str:
        """Combine graph facts and vector results into a single context string."""
        parts = []

        if graph_facts:
            parts.append('=== Knowledge Graph Facts ===')
            for fact in graph_facts:
                parts.append(f'  • {fact}')

        if vector_results.get('chunks'):
            parts.append('\n=== Relevant Text Passages ===')
            for i, chunk in enumerate(vector_results['chunks']):
                parts.append(f'  Passage {i+1}: {chunk}')

        return '\n'.join(parts)

    def _generate_answer(self, query: str, context: str) -> str:
        """Generate answer from context. Template-based for this lab."""
        # In production: send (query + context) to an LLM
        # For this lab: return the structured context as the answer
        return f"Question: {query}\n\nBased on the knowledge graph and documents:\n\n{context}"

    def query(self, question: str, max_hops: int = 2,
             top_k_chunks: int = 3, top_k_nodes: int = 3) -> Dict:
        """Full hybrid query pipeline."""
        # Step 1: Graph retrieval
        graph_facts = self.graph_retriever.retrieve(question, max_hops)

        # Step 2: Vector retrieval
        vector_results = self.vector_retriever.retrieve(question, top_k_chunks, top_k_nodes)

        # Step 3: Build context
        context = self._build_context(graph_facts, vector_results)

        # Step 4: Generate answer
        answer = self._generate_answer(question, context)

        return {
            'question': question,
            'answer': answer,
            'graph_facts': graph_facts,
            'vector_chunks': vector_results['chunks'],
            'vector_nodes': vector_results['nodes'],
        }


print("HybridQueryEngine defined.")

HybridQueryEngine defined.


In [34]:
# Initialize and test
engine = HybridQueryEngine(graph_retriever, vector_retriever)

# Test with multiple queries
test_queries = [
    "Who at Acme knows Python?",
    "What does Alice Johnson work on?",
    "Who reports to Emily Chen?",
]

for q in test_queries:
    result = engine.query(q, max_hops=1, top_k_chunks=2, top_k_nodes=2)
    print("=" * 70)
    print(result['answer'])
    print()

Question: Who at Acme knows Python?

Based on the knowledge graph and documents:


=== Relevant Text Passages ===
  Passage 1: Acme Corp is a technology company founded in 2015 and headquartered in San Francisco. The company has three main departments: Engineering, Data Science, and Product. Emily Chen is the CTO and oversees all technical departments. The company has about 50 employees. Carol Martinez is a Senior Data Scientist at Acme Corp. She works in the Data Science department and has expertise in Python, SQL, Apache Spark, and statistical modeling. Carol joined in 2019 and leads ProjectZ.
  Passage 2: David knows Python and JavaScript. He is currently working on ProjectY, which is an internal dashboard tool.

Question: What does Alice Johnson work on?

Based on the knowledge graph and documents:

=== Knowledge Graph Facts ===
  • Alice Johnson has_role Senior Software Engineer
  • Acme Corp founded_year 2015
  • Bob Smith works_at Acme Corp
  • Emily Chen works_at Acme Corp
  • 

---
## Phase 4 Complete ✅

**What we built:**
- `GraphRetriever` — finds entities in query, traverses KG neighborhood (BFS with max_hops)
- `VectorRetriever` — searches ChromaDB for similar chunks + nodes
- `HybridQueryEngine` — combines both retrievers, builds context, generates answer

**What we have now:**
- `engine` — a working hybrid query engine that answers questions using graph + vector

**Next → Phase 5: Evaluation**
- Measure answer relevance, graph coverage, response time

---
# Phase 5: Evaluation

**Goal:** Measure how well our KG RAG pipeline performs.

```
Three metrics:

1. Answer Relevance  — does the retrieved context match the question?
   (cosine similarity between query embedding and context embedding)

2. Graph Coverage    — how much of the KG did we use?
   (% of graph entities that appear in the answer)

3. Response Time     — how fast is the pipeline?
   (wall-clock time for the full query)
```

## 5.1 Evaluation Metrics

We compute three scores for each query to understand pipeline quality.

In [35]:
import time
import numpy as np

class Evaluator:
    """Evaluates KG RAG pipeline quality."""

    def __init__(self, embed_model: EmbeddingModel, graph: nx.DiGraph):
        self.embed_model = embed_model
        self.graph = graph

    def answer_relevance(self, query: str, context: str) -> float:
        """Cosine similarity between query and context embeddings (0 to 1)."""
        embs = self.embed_model.embed([query, context])
        q_vec, c_vec = np.array(embs[0]), np.array(embs[1])
        cos_sim = np.dot(q_vec, c_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(c_vec))
        return float(cos_sim)

    def graph_coverage(self, graph_facts: List[str]) -> float:
        """Fraction of graph nodes that appear in retrieved facts."""
        if not graph_facts:
            return 0.0
        fact_text = ' '.join(graph_facts).lower()
        total_nodes = self.graph.number_of_nodes()
        covered = sum(1 for n in self.graph.nodes() if n.lower() in fact_text)
        return covered / total_nodes if total_nodes > 0 else 0.0

    def evaluate_query(self, engine: HybridQueryEngine, question: str) -> Dict:
        """Run a query and compute all metrics."""
        start = time.time()
        result = engine.query(question)
        elapsed = time.time() - start

        relevance = self.answer_relevance(question, result['answer'])
        coverage = self.graph_coverage(result['graph_facts'])

        return {
            'question': question,
            'relevance': round(relevance, 3),
            'graph_coverage': round(coverage, 3),
            'response_time_ms': round(elapsed * 1000, 1),
            'num_graph_facts': len(result['graph_facts']),
            'num_chunks': len(result['vector_chunks']),
        }


print("Evaluator defined.")

Evaluator defined.


## 5.2 Run Evaluation

Test with a set of representative queries and display a results table.

In [36]:
evaluator = Evaluator(embed_model, kg)

eval_queries = [
    "Who at Acme knows Python?",
    "What does Alice Johnson work on?",
    "Who reports to Emily Chen?",
    "What skills does ProjectX require?",
    "Which department does Bob Smith manage?",
]

print(f"{'Query':<45} {'Relevance':>10} {'Coverage':>10} {'Time(ms)':>10} {'Facts':>6} {'Chunks':>7}")
print('-' * 95)

all_metrics = []
for q in eval_queries:
    m = evaluator.evaluate_query(engine, q)
    all_metrics.append(m)
    print(f"{m['question']:<45} {m['relevance']:>10.3f} {m['graph_coverage']:>10.3f} {m['response_time_ms']:>10.1f} {m['num_graph_facts']:>6} {m['num_chunks']:>7}")

# Averages
avg_rel = sum(m['relevance'] for m in all_metrics) / len(all_metrics)
avg_cov = sum(m['graph_coverage'] for m in all_metrics) / len(all_metrics)
avg_time = sum(m['response_time_ms'] for m in all_metrics) / len(all_metrics)
print('-' * 95)
print(f"{'AVERAGE':<45} {avg_rel:>10.3f} {avg_cov:>10.3f} {avg_time:>10.1f}")

Query                                          Relevance   Coverage   Time(ms)  Facts  Chunks
-----------------------------------------------------------------------------------------------
Who at Acme knows Python?                          0.658      0.000      455.7      0       3
What does Alice Johnson work on?                   0.530      1.000       30.9     12       3
Who reports to Emily Chen?                         0.610      1.000       28.6     12       3
What skills does ProjectX require?                 0.680      0.000       53.0      0       3
Which department does Bob Smith manage?            0.531      1.000       37.4     12       3
-----------------------------------------------------------------------------------------------
AVERAGE                                            0.602      0.600      121.1


---
## Phase 5 Complete ✅

**What we built:**
- `Evaluator` — computes answer relevance (cosine sim), graph coverage (% nodes used), response time

**What we measured:**
- Relevance: how well the context matches the question (higher = better)
- Coverage: what fraction of the KG was used (higher = more comprehensive)
- Time: end-to-end latency in milliseconds

**Next → Phase 6: Visualization**
- Interactive graph visualization with PyVis
- Highlight query paths in the graph

---
# Phase 6: Visualization

**Goal:** Create an interactive graph visualization using PyVis — see the KG in a browser.

```
NetworkX Graph (data structure)
        │
        ▼
   PyVis converts to interactive HTML
   (nodes are draggable, edges show labels on hover)
        │
        ▼
   knowledge_graph.html  ← open in browser
```

Nodes are color-coded by type: Person=blue, Project=green, Skill=orange, etc.

## 6.1 Graph Visualizer

Converts our NetworkX graph into an interactive PyVis HTML file.

In [37]:
from pyvis.network import Network

class KGVisualizer:
    """Interactive Knowledge Graph visualization using PyVis."""

    TYPE_COLORS = {
        'Person': '#4A90D9',      # blue
        'Company': '#E74C3C',     # red
        'Project': '#2ECC71',     # green
        'Skill': '#F39C12',       # orange
        'Department': '#9B59B6',  # purple
        'Other': '#95A5A6',       # gray
    }

    def __init__(self, graph: nx.DiGraph):
        self.graph = graph

    def visualize(self, output_path: str = 'knowledge_graph.html',
                  highlight_nodes: List[str] = None,
                  width: str = '100%', height: str = '700px') -> str:
        """Generate interactive HTML visualization."""
        net = Network(height=height, width=width, directed=True, notebook=True)
        net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)

        highlight_set = set(highlight_nodes or [])

        # Add nodes
        for node, data in self.graph.nodes(data=True):
            node_type = data.get('type', 'Other')
            color = self.TYPE_COLORS.get(node_type, '#95A5A6')
            size = 30 if node in highlight_set else 20
            border_width = 4 if node in highlight_set else 1
            net.add_node(
                node, label=node, color=color, size=size,
                title=f"{node} ({node_type})",
                borderWidth=border_width
            )

        # Add edges
        for u, v, data in self.graph.edges(data=True):
            net.add_edge(u, v, label=data['predicate'], title=data['predicate'])

        net.save_graph(output_path)
        return output_path


print("KGVisualizer defined.")

KGVisualizer defined.


In [38]:
# Generate the full graph visualization
visualizer = KGVisualizer(kg)
output_file = visualizer.visualize('knowledge_graph.html')
print(f"Full graph saved to: {output_file}")
print(f"Open in browser to explore interactively.")

Full graph saved to: knowledge_graph.html
Open in browser to explore interactively.


## 6.2 Query Path Visualization

Highlight the nodes and edges that were used to answer a specific query.
This helps debug and explain *why* the system gave a particular answer.

In [39]:
# Visualize a query path
query = "Who at Acme knows Python?"
result = engine.query(query, max_hops=1)

# Extract entity names from graph facts
highlight = set()
for fact in result['graph_facts']:
    for node in kg.nodes():
        if node in fact:
            highlight.add(node)

print(f"Query: '{query}'")
print(f"Highlighted nodes: {highlight}\n")

# Generate highlighted visualization
query_file = visualizer.visualize('query_path.html', highlight_nodes=list(highlight))
print(f"Query path saved to: {query_file}")
print("Highlighted nodes appear larger with thick borders.")

Query: 'Who at Acme knows Python?'
Highlighted nodes: set()

Query path saved to: query_path.html
Highlighted nodes appear larger with thick borders.


## 6.3 Graph Statistics Summary

Final overview of the complete knowledge graph.

In [40]:
from collections import Counter

print("=== Knowledge Graph Statistics ===\n")
print(f"Total nodes:  {kg.number_of_nodes()}")
print(f"Total edges:  {kg.number_of_edges()}")

# Node types
type_counts = Counter(d.get('type', 'Other') for _, d in kg.nodes(data=True))
print(f"\nNodes by type:")
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")

# Edge types
edge_counts = Counter(d['predicate'] for _, _, d in kg.edges(data=True))
print(f"\nEdges by predicate:")
for p, c in edge_counts.most_common():
    print(f"  {p}: {c}")

# Most connected nodes
print(f"\nMost connected nodes (by degree):")
degrees = sorted(kg.degree(), key=lambda x: x[1], reverse=True)
for node, deg in degrees[:5]:
    print(f"  {node}: {deg} connections")

=== Knowledge Graph Statistics ===

Total nodes:  13
Total edges:  12

Nodes by type:
  Person: 10
  Company: 3

Edges by predicate:
  has_role: 5
  works_at: 5
  headquartered_in: 1
  founded_year: 1

Most connected nodes (by degree):
  Acme Corp: 7 connections
  Alice Johnson: 2 connections
  Bob Smith: 2 connections
  David Lee: 2 connections
  Carol Martinez: 2 connections


---
## Phase 6 Complete ✅

**What we built:**
- `KGVisualizer` — converts NetworkX graph to interactive PyVis HTML
- Color-coded nodes by entity type
- Query path highlighting (larger nodes with thick borders)
- Graph statistics summary

**Output files:**
- `knowledge_graph.html` — full interactive graph
- `query_path.html` — graph with query-relevant nodes highlighted

---
# ✅ All 6 Phases Complete!

```
Phase 1: Data Ingestion          ✅ Load, clean, chunk documents
Phase 2: KG Construction         ✅ Extract triplets, resolve entities, build graph
Phase 3: Embedding & Indexing    ✅ Embed nodes + chunks, store in ChromaDB
Phase 4: Query Engine            ✅ Hybrid retrieval (graph + vector), generate answer
Phase 5: Evaluation              ✅ Relevance scoring, coverage metrics, response time
Phase 6: Visualization           ✅ Interactive PyVis graph, query path display
```

**Classes built (10 total):**

| Phase | Class | Purpose |
|-------|-------|---------|
| 1 | `TextCleaner` | Normalize raw text |
| 1 | `DocumentLoader` | Load from dicts/files |
| 1 | `TextChunker` | Split with overlap |
| 1 | `DataIngestionPipeline` | Phase 1 wrapper |
| 2 | `RuleBasedTripletExtractor` | 12 regex patterns |
| 2 | `EntityResolver` | Merge duplicate names |
| 2 | `TripletValidator` | Remove invalid/dupes |
| 2 | `KnowledgeGraphBuilder` | Build NetworkX graph |
| 2 | `KGConstructionPipeline` | Phase 2 wrapper |
| 3 | `EmbeddingModel` | sentence-transformers |
| 3 | `VectorStore` | ChromaDB wrapper |
| 3 | `EmbeddingIndexingPipeline` | Phase 3 wrapper |
| 4 | `GraphRetriever` | BFS graph traversal |
| 4 | `VectorRetriever` | ChromaDB search |
| 4 | `HybridQueryEngine` | Combine both |
| 5 | `Evaluator` | Relevance + coverage |
| 6 | `KGVisualizer` | PyVis interactive HTML |

**Data classes:** `Document`, `Chunk`, `Triplet`

**Tech stack:** NetworkX, ChromaDB, sentence-transformers, PyVis — all local, no API keys.